In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
# os.walk() is used to traverse through a directory and list all files and folders dynamically
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd


In [ ]:
import os

# Define the root directory
root_dir = "/content/drive/MyDrive/skin lesions/my_data"

# Initialize lists to store information
file_paths = []
disease_types = []
splits = []

# Walk through the directory and collect data
for split in os.listdir(root_dir):  # Test, Train, or Validation
    split_path = os.path.join(root_dir, split)
    if os.path.isdir(split_path):  # Ensure it's a directory
        for disease in os.listdir(split_path):  # Disease types
            disease_path = os.path.join(split_path, disease)
            if os.path.isdir(disease_path):  # Ensure it's a directory
                for file in os.listdir(disease_path):  # Files
                    file_path = os.path.join(disease_path, file)
                    if os.path.isfile(file_path):  # Ensure it's a file
                        file_paths.append(file_path)
                        disease_types.append(disease)
                        splits.append(split)

# Create the DataFrame
data = {
    "File Path": file_paths,
    "Disease Type": disease_types,
    "Split": splits,
    "File Name": [os.path.basename(fp) for fp in file_paths]
}

df = pd.DataFrame(data)

df


,File Path,Disease Type,Split,File Name
0,/content/drive/MyDrive/skin lesions/my_data/va...,Vascular lesions,val,ISIC_0026693.jpg
1,/content/drive/MyDrive/skin lesions/my_data/va...,Vascular lesions,val,ISIC_0026163.jpg
2,/content/drive/MyDrive/skin lesions/my_data/va...,Vascular lesions,val,ISIC_0024747.jpg
3,/content/drive/MyDrive/skin lesions/my_data/va...,Vascular lesions,val,ISIC_0024475.jpg
4,/content/drive/MyDrive/skin lesions/my_data/va...,Vascular lesions,val,ISIC_0027983.jpg
...,...,...,...,...
13297,/content/drive/MyDrive/skin lesions/my_data/te...,Actinic keratoses,test,ISIC_0068648.jpg
13298,/content/drive/MyDrive/skin lesions/my_data/te...,Actinic keratoses,test,ISIC_0072646.jpg
13299,/content/drive/MyDrive/skin lesions/my_data/te...,Actinic keratoses,test,ISIC_0069211.jpg
13300,/content/drive/MyDrive/skin lesions/my_data/te...,Actinic keratoses,test,ISIC_0067755.jpg


In [ ]:
df["File Path"][0]

'/content/drive/MyDrive/skin lesions/my_data/val/Vascular lesions/ISIC_0026693.jpg'

In [ ]:
#####for splitting data into test, train, val after preprocessing stage followed by normalizing them
import os
import numpy as np
from PIL import Image #For image processing (reading, resizing, etc.)
from tqdm import tqdm  # to show progress bar

# Define the root directory
root_dir = "/content/drive/MyDrive/skin lesions/my_data"

# Image processing parameters
target_size = (64, 64)  # Resize target
normalize_factor = 255.0  # Normalize by dividing pixel values by 255

# Dictionary to store image and label data for each split
data = {
    "train": {"images": [], "labels": []},
    "test": {"images": [], "labels": []},
    "val": {"images": [], "labels": []}
}

# Walk through the directory and process images
for split in os.listdir(root_dir):  # E.g., train, test, val
    split_path = os.path.join(root_dir, split)
    if split in data and os.path.isdir(split_path):  # Ensure it's a directory and a valid split
        for disease in os.listdir(split_path):  # Disease types
            disease_path = os.path.join(split_path, disease)
            if os.path.isdir(disease_path):  # Ensure it's a directory
                for file in tqdm(os.listdir(disease_path), desc=f"Processing {disease} in {split}"):
                    file_path = os.path.join(disease_path, file)
                    if os.path.isfile(file_path):  # Ensure it's a file
                        # Open and process the image
                        img = Image.open(file_path).convert("RGB")  # Convert to RGB
                        img_resized = img.resize(target_size)  # Resize
                        img_array = np.array(img_resized) / normalize_factor  # Normalize

                        # Append to the corresponding split
                        data[split]["images"].append(img_array)
                        data[split]["labels"].append(disease)

# Save the processed data for each split
for split in data:
    images = np.array(data[split]["images"], dtype=np.float32)
    labels = np.array(data[split]["labels"])

    np.save(f"{split}_images.npy", images)
    np.save(f"{split}_labels.npy", labels)

    print(f"Processed {len(images)} images for {split}. Data saved as '{split}_images.npy' and '{split}_labels.npy'.")



Processing Chickenpox in train: 100%|██████████| 900/900 [00:22<00:00, 39.27it/s] 
Processing Benign keratosis-like lesions in train: 100%|██████████| 1000/1000 [00:36<00:00, 27.67it/s]
Processing Actinic keratoses in test: 100%|██████████| 88/88 [00:19<00:00,  4.44it/s]


Processed 10940 images for train. Data saved as 'train_images.npy' and 'train_labels.npy'.
Processed 1185 images for test. Data saved as 'test_images.npy' and 'test_labels.npy'.
Processed 1177 images for val. Data saved as 'val_images.npy' and 'val_labels.npy'.


In [ ]:
import torch  # PyTorch library for deep learning operations.
import torch.nn as nn  # Neural network layers, loss functions, etc.
import torch.optim as optim  # Optimization algorithms like Adam
from torch.utils.data import DataLoader, Dataset  # Utilities for custom datasets and batching
from torchvision import models, transforms  # Pre-trained models and transformations
import numpy as np

# Load Data
train_images = np.load("/content/train_images.npy")
train_labels = np.load("/content/train_labels.npy")
val_images = np.load("/content/val_images.npy")
val_labels = np.load("/content/val_labels.npy")

# Define Dataset
"""
SkinLesionDataset loads images and labels.
- images: NumPy array of image data
- labels: string labels (e.g., 'melanoma', 'nevus', etc.)
- transform: preprocessing applied to each image
- label_to_idx: map from label string → integer
- idx_to_label: reverse mapping
"""
class SkinLesionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
        self.label_to_idx = {label: idx for idx, label in enumerate(np.unique(labels))}
        self.idx_to_label = {idx: label for label, idx in self.label_to_idx.items()}
        self.labels = [self.label_to_idx[label] for label in labels]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize to [-1, 1]
])

# Create Datasets
train_dataset = SkinLesionDataset(train_images, train_labels, transform=transform)
val_dataset = SkinLesionDataset(val_images, val_labels, transform=transform)

# Data Loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Define ResNet Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)

# Adjust first conv layer for 64x64 images
model.conv1 = nn.Conv2d(
    in_channels=3,
    out_channels=64,
    kernel_size=(7, 7),
    stride=(2, 2),
    padding=(3, 3),
    bias=False
)

# Change final fully connected layer
model.fc = nn.Linear(model.fc.in_features, len(np.unique(train_labels)))

model = model.to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Explanation:
        # outputs.max(1) → returns (max_value, max_index)
        # max_index = predicted class
        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_accuracy = 100. * correct / total
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {train_loss:.4f}, Accuracy: {train_accuracy:.2f}%")

    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            # Same max() explanation applies here
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_accuracy = 100. * correct / total
    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")

# Save the Trained Model
torch.save(model.state_dict(), "resnet_skin_lesion.pth")
print("Model saved as resnet_skin_lesion.pth")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 96.0MB/s]


Epoch 1/10, Loss: 574.1624, Accuracy: 42.48%
Validation Loss: 51.9948, Validation Accuracy: 51.23%
Epoch 2/10, Loss: 373.5138, Accuracy: 61.38%
Validation Loss: 45.4858, Validation Accuracy: 59.90%
Epoch 3/10, Loss: 297.9889, Accuracy: 68.98%
Validation Loss: 31.6850, Validation Accuracy: 71.20%
Epoch 4/10, Loss: 247.7485, Accuracy: 74.20%
Validation Loss: 28.7291, Validation Accuracy: 72.05%
Epoch 5/10, Loss: 212.3229, Accuracy: 77.31%
Validation Loss: 37.7206, Validation Accuracy: 67.37%
Epoch 6/10, Loss: 185.5083, Accuracy: 80.33%
Validation Loss: 29.8655, Validation Accuracy: 74.51%
Epoch 7/10, Loss: 153.3008, Accuracy: 83.55%
Validation Loss: 30.2335, Validation Accuracy: 73.58%
Epoch 8/10, Loss: 134.1080, Accuracy: 86.41%
Validation Loss: 30.9427, Validation Accuracy: 74.00%
Epoch 9/10, Loss: 109.2742, Accuracy: 88.84%
Validation Loss: 32.9542, Validation Accuracy: 75.19%
Epoch 10/10, Loss: 90.4997, Accuracy: 90.92%
Validation Loss: 36.2001, Validation Accuracy: 73.32%
Model save

Mobile Net


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import numpy as np

# Load Data
train_images = np.load("/content/train_images.npy")
train_labels = np.load("/content/train_labels.npy")
val_images = np.load("/content/val_images.npy")
val_labels = np.load("/content/val_labels.npy")

# Dataset
class SkinLesionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.transform = transform
        self.label_to_idx = {label: idx for idx, label in enumerate(np.unique(labels))}
        self.idx_to_label = {idx: label for label, idx in self.label_to_idx.items()}
        self.labels = [self.label_to_idx[label] for label in labels]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Datasets + Loaders
train_dataset = SkinLesionDataset(train_images, train_labels, transform)
val_dataset = SkinLesionDataset(val_images, val_labels, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# ===========================================
# ✅ MobileNetV2 MODEL (replaces ResNet18)
# ===========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.mobilenet_v2(pretrained=True)

# Change classifier to match number of classes
num_classes = len(np.unique(train_labels))
model.classifier[1] = nn.Linear(model.last_channel, num_classes)

model = model.to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_acc = 100. * correct / total
    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.2f}%")

# Save Model
torch.save(model.state_dict(), "mobilenet_skin_lesion.pth")
print("Model saved as mobilenet_skin_lesion.pth")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 83.8MB/s]


Epoch 1/10, Loss: 430.2976, Accuracy: 56.97%
Validation Loss: 32.4914, Validation Accuracy: 66.53%
Epoch 2/10, Loss: 290.6265, Accuracy: 69.57%
Validation Loss: 28.4758, Validation Accuracy: 72.98%
Epoch 3/10, Loss: 253.7607, Accuracy: 72.95%
Validation Loss: 28.8644, Validation Accuracy: 71.28%
Epoch 4/10, Loss: 231.2525, Accuracy: 75.26%
Validation Loss: 27.3821, Validation Accuracy: 74.94%
Epoch 5/10, Loss: 205.4398, Accuracy: 77.71%
Validation Loss: 23.0172, Validation Accuracy: 78.16%
Epoch 6/10, Loss: 186.8452, Accuracy: 80.27%
Validation Loss: 24.6918, Validation Accuracy: 77.15%
Epoch 7/10, Loss: 177.4659, Accuracy: 81.25%
Validation Loss: 25.1481, Validation Accuracy: 77.23%
Epoch 8/10, Loss: 159.1828, Accuracy: 83.33%
Validation Loss: 28.6146, Validation Accuracy: 74.77%
Epoch 9/10, Loss: 152.3831, Accuracy: 83.85%
Validation Loss: 26.3147, Validation Accuracy: 75.79%
Epoch 10/10, Loss: 142.4857, Accuracy: 84.95%
Validation Loss: 31.1061, Validation Accuracy: 74.94%
Model sav

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import numpy as np

# ---------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = len(np.unique(np.load("/content/train_labels.npy")))  # for label count

model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)

model.load_state_dict(torch.load("mobilenet_skin_lesion.pth", map_location=device))
model = model.to(device)
model.eval()

# ---------------------------------------------------
# TRANSFORM (same as training)
# ---------------------------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# ---------------------------------------------------
# LABEL MAPPING (rebuild dictionary)
# ---------------------------------------------------
train_labels = np.load("/content/train_labels.npy")
unique_labels = np.unique(train_labels)
label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
idx_to_label = {v: k for k, v in label_to_idx.items()}

# ---------------------------------------------------
# PREDICTION FUNCTION
# ---------------------------------------------------
def predict(image):
    image = image.resize((64, 64))               # resize to training input
    img_tensor = transform(image).unsqueeze(0)   # add batch dimension
    img_tensor = img_tensor.to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        confidence, predicted_class = torch.max(probabilities, dim=0)

    predicted_label = idx_to_label[predicted_class.item()]
    confidence = confidence.item()

    return {
        "Predicted Class": predicted_label,
        "Confidence": float(round(confidence * 100, 2))
    }

# ---------------------------------------------------
# GRADIO UI
# ---------------------------------------------------
interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload a skin lesion image"),
    outputs=[
        gr.JSON(label="Prediction Result")
    ],
    title="Skin Lesion Classifier (MobileNetV2)",
    description="Upload a 64×64 lesion image and the model will classify it."
)

interface.launch()


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://df628082191f62d229.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ensemble

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import numpy as np

# ---------------------------------------------------
# Load Data
# ---------------------------------------------------
train_images = np.load("/content/train_images.npy")
train_labels = np.load("/content/train_labels.npy")
val_images = np.load("/content/val_images.npy")
val_labels = np.load("/content/val_labels.npy")

# ---------------------------------------------------
# Dataset
# ---------------------------------------------------
class SkinLesionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
        self.label_to_idx = {label: idx for idx, label in enumerate(np.unique(labels))}
        self.idx_to_label = {idx: label for label, idx in self.label_to_idx.items()}
        self.labels = [self.label_to_idx[label] for label in labels]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# ---------------------------------------------------
# Transforms
# ---------------------------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Dataloaders
train_dataset = SkinLesionDataset(train_images, train_labels, transform=transform)
val_dataset = SkinLesionDataset(val_images, val_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# ---------------------------------------------------
# Device
# ---------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = len(np.unique(train_labels))

# ---------------------------------------------------
# MODEL 1: ResNet18
# ---------------------------------------------------
resnet = models.resnet18(pretrained=True)

resnet.conv1 = nn.Conv2d(
    3, 64, kernel_size=7, stride=2, padding=3, bias=False
)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = resnet.to(device)

# ---------------------------------------------------
# MODEL 2: MobileNetV2
# ---------------------------------------------------
mobilenet = models.mobilenet_v2(pretrained=True)
mobilenet.classifier[1] = nn.Linear(mobilenet.last_channel, num_classes)
mobilenet = mobilenet.to(device)

# ---------------------------------------------------
# Loss and Optimizers
# ---------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    list(resnet.parameters()) + list(mobilenet.parameters()),
    lr=0.001
)

# ---------------------------------------------------
# Training Loop
# ---------------------------------------------------
num_epochs = 10

for epoch in range(num_epochs):
    resnet.train()
    mobilenet.train()

    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward passes
        out_res = resnet(images)
        out_mob = mobilenet(images)

        # Ensemble logits (simple average)
        ensemble_logits = (out_res + out_mob) / 2

        # Loss on ensemble output
        loss = criterion(ensemble_logits, labels)

        # Backprop updates BOTH MODELS
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = ensemble_logits.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {train_loss:.4f} | Accuracy: {train_acc:.2f}%")

    # ---------------------------------------------------
    # Validation
    # ---------------------------------------------------
    resnet.eval()
    mobilenet.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            out_res = resnet(images)
            out_mob = mobilenet(images)

            ensemble_logits = (out_res + out_mob) / 2

            loss = criterion(ensemble_logits, labels)
            val_loss += loss.item()

            _, predicted = ensemble_logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    val_acc = 100 * correct / total
    print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.2f}%")
    print("------------------------------------------------------")

# ---------------------------------------------------
# Save Ensemble (Both Models in ONE File)
# ---------------------------------------------------
torch.save({
    "resnet": resnet.state_dict(),
    "mobilenet": mobilenet.state_dict()
}, "ensemble_trained.pth")

print("Ensemble model saved as ensemble_trained.pth")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-d

Epoch 1/10 | Loss: 428.7425 | Accuracy: 57.16%
Validation Loss: 32.6771 | Validation Accuracy: 68.14%
------------------------------------------------------
Epoch 2/10 | Loss: 303.6204 | Accuracy: 68.11%
Validation Loss: 48.8313 | Validation Accuracy: 66.27%
------------------------------------------------------
Epoch 3/10 | Loss: 254.6299 | Accuracy: 73.46%
Validation Loss: 26.9546 | Validation Accuracy: 73.83%
------------------------------------------------------
Epoch 4/10 | Loss: 220.6521 | Accuracy: 76.75%
Validation Loss: 31.4822 | Validation Accuracy: 73.15%
------------------------------------------------------
Epoch 5/10 | Loss: 210.7407 | Accuracy: 77.67%
Validation Loss: 25.5239 | Validation Accuracy: 77.57%
------------------------------------------------------
Epoch 6/10 | Loss: 195.4778 | Accuracy: 79.31%
Validation Loss: 26.3390 | Validation Accuracy: 74.94%
------------------------------------------------------
Epoch 7/10 | Loss: 172.0988 | Accuracy: 81.35%
Validation 

In [12]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np

# ---------------------------------------------------
# DEVICE
# ---------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------
# LABEL LOADING
# ---------------------------------------------------
train_labels = np.load("/content/train_labels.npy")
unique_labels = np.unique(train_labels)
idx_to_label = {i: unique_labels[i] for i in range(len(unique_labels))}
num_classes = len(unique_labels)

# ---------------------------------------------------
# REBUILD RESNET MODEL
# ---------------------------------------------------
resnet = models.resnet18(pretrained=False)
resnet.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = resnet.to(device)
resnet.eval()

# ---------------------------------------------------
# REBUILD MOBILENET MODEL
# ---------------------------------------------------
mobilenet = models.mobilenet_v2(pretrained=False)
mobilenet.classifier[1] = nn.Linear(mobilenet.last_channel, num_classes)
mobilenet = mobilenet.to(device)
mobilenet.eval()

# ---------------------------------------------------
# LOAD CHECKPOINT (BOTH MODELS)
# ---------------------------------------------------
checkpoint = torch.load("/content/ensemble_trained.pth", map_location=device)

resnet.load_state_dict(checkpoint["resnet"])
mobilenet.load_state_dict(checkpoint["mobilenet"])

print("✔ Ensemble Models Loaded Successfully!")

# ---------------------------------------------------
# TRANSFORM
# ---------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# ---------------------------------------------------
# PREDICTION FUNCTION
# ---------------------------------------------------
def predict(image):
    image = image.convert("RGB")
    img_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():

        # model outputs
        res_logits = resnet(img_tensor)
        mob_logits = mobilenet(img_tensor)

        # ensemble = average logits
        final_logits = (res_logits + mob_logits) / 2

        probabilities = torch.softmax(final_logits, dim=1)[0].cpu().numpy()
        pred_idx = np.argmax(probabilities)
        pred_label = idx_to_label[pred_idx]

    # prepare return dict
    prob_dict = {idx_to_label[i]: float(probabilities[i]) for i in range(len(probabilities))}

    return pred_label, prob_dict

# ---------------------------------------------------
# GRADIO UI
# ---------------------------------------------------
ui = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Label(label="Predicted Class"),
        gr.Label(label="Class Probabilities")
    ],
    title="Skin Lesion Ensemble Classifier (ResNet18 + MobileNetV2)",
    description="This model uses an ensemble of two CNNs for more accurate predictions."
)

ui.launch()


✔ Ensemble Models Loaded Successfully!
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59ac58463d5e09e838.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
